# B+ Bäume — Übungsblatt

**Algorithmen und Datenstrukturen SS26 · Till Hänisch DHBW Heidenheim**

## Lernziele

Sie evaluieren das Laufzeitverhalten einer B+ Baum-Implementierung empirisch und
setzen die Messungen in Bezug zur Theorie aus der Vorlesung. Konkret untersuchen Sie,
wie die Seitengröße ($B$) und die Größe des Buffer-Pools die Anzahl logischer
Knotenzugriffe und echter Disk-Reads beeinflussen.

## Material

- `BPlusTree.py` — In-Memory-Variante
- `BPlusTree_io.py` — mit simuliertem Buffer-Pool

## Abgabe

Füllen Sie dieses Notebook aus (alle Zellen ausgeführt, Plots sichtbar, Antworten
auf Reflexionsfragen als Markdown an der markierten Stelle) und geben Sie die
`.ipynb`-Datei ab.

---
## Setup

In [ ]:
import random
import time
import numpy as np
import matplotlib.pyplot as plt

from BPlusTree import BPlusTree as BPlusTreeMem
from BPlusTreeIO import BPlusTree as BPlusTreeIO

random.seed(0)
np.random.seed(0)
%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['figure.dpi'] = 110

---
## Aufgabe 1 — Erste Messungen (Warmup)

*Richtzeit: 10 Min. &nbsp;·&nbsp; Ergebnis: Kurze Notizen zu den Beobachtungen.* 1 Punkt

Führen Sie den Benchmark in der nächsten Zelle aus und betrachten Sie die ausgegebene Tabelle.

In [ ]:
# Mini-Benchmark wie im __main__ von bplustree.py.
random.seed(42)
N = 100_000
data = [(random.randint(0, 10_000_000), i) for i in range(N)]

print(f"{'page_size':>10} {'height':>7} {'nodes':>8} {'avg accesses':>14}")
for B in (4, 8, 16, 32, 64, 128, 256, 512):
    t = BPlusTreeMem(page_size=B)
    for k, v in data:
        t.insert(k, v)
    t.reset_stats()
    for k, _ in data[:10_000]:
        t.search(k)
    avg = t.node_accesses / 10_000
    print(f"{B:>10} {t.height():>7} {t.count_nodes():>8} {avg:>14.2f}")

**Fragen:**
1. Wie verändert sich die *Baumhöhe*, wenn `page_size` verdoppelt wird? Passt das zur Formel $\mathcal{O}(\log_B N)$?
2. Die Spalte `avg accesses` zeigt für jede Konfiguration einen glatten ganzzahligen Wert. Warum ist das so?
3. Notieren Sie die Zahl der Knoten für `page_size=4` und `512`. Um welchen Faktor unterscheidet sie sich?

> **Ihre Antworten (Doppelklick, um zu editieren):**
>
> *1. …*
>
> *2. baum perfekt ausgeglichen
>
> *3. um den Faktor 180 (512/4=128, aber ich spare mir ja auch viele interne Blätter)

---
## Aufgabe 2 — Seitengröße systematisch variieren

*Richtzeit: 25 Min. &nbsp;·&nbsp; Ergebnis: Zwei Plots + kurze Diskussion.* 1 Punkt

Sie untersuchen, wie sich `page_size` auf Suchkosten und Build-Zeit auswirkt. Die
Messschleife ist unten vorbereitet — ergänzen Sie die beiden Plots.

In [ ]:
# Messung: verschiedene Seitengrößen durchprobieren.
random.seed(0)
N = 100_000
data = [(random.randint(0, 10_000_000), i) for i in range(N)]
queries = [k for k, _ in random.sample(data, 10_000)]

sizes, accesses, build_times = [], [], []
for B in [4, 8, 16, 32, 64, 128, 256, 512, 1024]:
    t = BPlusTreeMem(page_size=B)

    t0 = time.perf_counter()
    for k, v in data:
        t.insert(k, v)
    build_times.append(time.perf_counter() - t0)

    t.reset_stats()
    for k in queries:
        t.search(k)

    sizes.append(B)
    accesses.append(t.node_accesses / len(queries))

print("Messung fertig. Daten in `sizes`, `accesses`, `build_times`.")

In [ ]:
# TODO: Plot 1 — avg node accesses vs page_size (x-Achse logarithmisch!)
#       plt.semilogx(...)  plt.xlabel(...)  plt.ylabel(...)  plt.grid(True)  plt.show()



In [ ]:
# TODO: Plot 2 — build_time vs page_size



**Reflexion:**
- Plot 1 sollte mit wachsendem $B$ gegen einen *ganzzahligen Grenzwert* konvergieren — welchen, und warum?
- Plot 2 ist **nicht** monoton fallend. Warum können sehr große Seiten die Build-Phase wieder verlangsamen? (Tipp: was macht `bisect` in einer Seite mit 1024 Keys, was der `list.insert` danach?)

> *Ihre Antwort:*
>
> *…*

---
## Aufgabe 3 — Der Buffer-Pool

*Richtzeit: 45 Min. &nbsp;·&nbsp; Ergebnis: Plot + Diskussion + eine Rechnung.* 1 Punkt

Wir wechseln zu `BPlusTree_io.py`, das einen Buffer-Pool mit fester Kapazität
simuliert und *echte* Disk-Reads zählt.

### 3a) Baseline ermitteln

Zuerst brauchen wir zwei Referenzzahlen: wie groß ist der Baum insgesamt, und
wie viele davon sind interne Knoten?

In [ ]:
# Referenz-Baum mit praktisch unbegrenztem Buffer.
ref = BPlusTreeIO(page_size=64, buffer_capacity=10**9)
for k, v in data:
    ref.insert(k, v)

total_pages = ref.bm.total_pages()

internal_count = 0
stack = [ref.root_pid]
while stack:
    n = ref.bm.get(stack.pop())
    if not n.is_leaf:
        internal_count += 1
        stack.extend(n.children)

print(f"Gesamtseitenzahl : {total_pages}")
print(f"  davon intern   : {internal_count}")
print(f"  davon Blätter  : {total_pages - internal_count}")

### 3b) Buffer-Sweep

Variieren Sie `buffer_capacity` über mindestens 8 Werte von 4 bis
$\ge$ `total_pages`. Messen Sie für jede Kapazität `disk_reads` pro Query bei
10 000 zufälligen Lookups. **Stats nach dem Build zurücksetzen!**

In [ ]:
# TODO: Liste der zu testenden Kapazitäten. Tipp: grobe log-Skala,
# mit den Schwellen `internal_count` und `total_pages` als Ankern.
capacities = []  # z.B. [4, 16, 64, internal_count + 20, 256, 1024, total_pages + 10]

reads_per_q = []

for cap in capacities:
    # TODO:
    # 1) neuen BPlusTreeIO(page_size=64, buffer_capacity=cap) anlegen
    # 2) data einfügen
    # 3) t.bm.reset_stats()
    # 4) alle queries ausführen
    # 5) t.bm.disk_reads / len(queries) in reads_per_q anhängen
    pass

In [ ]:
# TODO: Plot — disk_reads/query vs buffer_capacity (x-Achse logarithmisch).
# Markieren Sie zusätzlich die zwei theoretischen Schwellen als vertikale Linien:
#   plt.axvline(internal_count, ls='--', color='orange', label='interne Knoten')
#   plt.axvline(total_pages,    ls='--', color='green',  label='gesamter Baum')



**Reflexion:**
- Bei welcher Buffer-Größe fällt `disk_reads/query` unter 1? Passt das zu `internal_count`?
- Wo geht die Kurve gegen 0? Passt das zu `total_pages`?
- Wo gibt es Abweichungen — und was ist die Ursache?

> *Ihre Antwort:*
>
> *…*

### 3c) Sortierte Queries

Wiederholen Sie den Messpunkt bei `buffer_capacity=16` mit einer **sortierten**
Query-Folge. Begründen Sie, wieso der Wert so viel niedriger liegt — und welcher
realen DB-Operation dieses Muster entspricht.

In [ ]:
# TODO: sortierte Queries, buffer_capacity=16, disk_reads/query ausgeben
# und mit dem random-Wert aus 3b vergleichen.



> *Ihre Begründung:*
>
> *…*